# Week 11 Lab — Clustering & Dimensionality Reduction

**Goal:** Apply PCA and K-Means to 114,000 Spotify tracks. Discover what structure emerges from audio features alone — without using the genre label.

By the end of this lab you will:
1. Scale features and understand why scaling matters for distance-based methods
2. Apply PCA to reduce 10 dimensions to 2–3 principal components
3. Use the elbow method and silhouette score to pick *K* for K-Means
4. Visualize clusters with t-SNE and compare them to genre labels

## Part 1: Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.stat import Correlation

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

spark = SparkSession.builder.appName("Clustering_Lab").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

KeyboardInterrupt: 

## Part 2: Load & Explore the Data

In [ ]:
df = spark.read.csv("spotify.csv", header=True, inferSchema=True)
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

### Audio Features

We will use 10 continuous audio features for clustering. These describe *what the music sounds like* — not metadata like artist name or popularity.

| Feature | Range | What it measures |
|---------|-------|------------------|
| danceability | 0–1 | How suitable for dancing |
| energy | 0–1 | Intensity and activity |
| loudness | dB (negative) | Overall volume |
| speechiness | 0–1 | Presence of spoken words |
| acousticness | 0–1 | Acoustic vs. electronic |
| instrumentalness | 0–1 | No vocals predicted |
| liveness | 0–1 | Audience presence detected |
| valence | 0–1 | Musical positivity |
| tempo | BPM | Speed of the track |
| duration_ms | milliseconds | Length of the track |

We exclude `key` (categorical, 0–11), `mode` (binary), `time_signature` (categorical), `popularity` (not an audio feature), and `explicit` (boolean).

In [ ]:
audio_features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "duration_ms"
]

# Summary statistics for the audio features
df.select(audio_features).describe().show()

**Look at the ranges.** Most features are 0–1, but `loudness` is in decibels (negative values), `tempo` is in BPM (~60–200+), and `duration_ms` is in milliseconds (tens of thousands to millions). 

**Question:** If you run K-Means directly on these raw features, which features will dominate the distance calculations? Why is this a problem?

In [ ]:
# How many genres are in the dataset?
# YOUR CODE HERE
# genre_counts = df.

print(f"Number of genres: {genre_counts.count()}")
genre_counts.show(10)

### Drop nulls in audio features

VectorAssembler will fail on null values. Clean them out now.

In [ ]:
before = df.count()
df = df.na.drop(subset=audio_features)
after = df.count()
print(f"Rows before: {before:,} | After: {after:,} | Dropped: {before - after}")

### Focus on Acoustically Distinct Genres

114 genres is too noisy — pop and dance sound identical, alt-rock and rock overlap heavily. To see clear clusters, we pick 6 genres with very different audio profiles: **classical, death-metal, hip-hop, ambient, EDM, comedy**.

We keep the full dataset as `df_full` so we can build a genre map of all 114 genres later.

In [ ]:
# Save full dataset for genre map later
df_full = df

# Filter to 6 acoustically distinct genres
selected_genres = ["classical", "death-metal", "hip-hop", "ambient", "edm", "comedy"]
df = df.filter(F.col("track_genre").isin(selected_genres))

print(f"Filtered rows: {df.count():,}")
df.groupBy("track_genre").count().orderBy(F.desc("count")).show()

### Correlation Matrix

Before reducing dimensions, let's see which features are already correlated. Highly correlated features carry redundant information — PCA will compress them into fewer components.

In [ ]:
# YOUR CODE HERE:
# 1. Use VectorAssembler to combine audio_features into a single vector column "features_vec"
# 2. Use Correlation.corr() to compute the Pearson correlation matrix
# 3. Convert to a numpy array and display as a heatmap
#
# Hint: 
#   assembler = VectorAssembler(inputCols=audio_features, outputCol="features_vec")
#   vec_df = assembler.transform(df).select("features_vec")
#   corr_matrix = Correlation.corr(vec_df, "features_vec").head()[0]
#   corr_array = corr_matrix.toArray()



In [ ]:
# Visualize the correlation matrix as a heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_array, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(audio_features)))
ax.set_yticks(range(len(audio_features)))
ax.set_xticklabels(audio_features, rotation=45, ha="right")
ax.set_yticklabels(audio_features)
plt.colorbar(im)
plt.title("Audio Feature Correlations")
plt.tight_layout()
plt.show()

**Interpret:** Which pairs of features show strong positive or negative correlation? Does energy–loudness or energy–acousticness stand out? These correlated features are exactly what PCA will compress.

## Part 3: Feature Preparation — Assemble & Scale

Both PCA and K-Means are **distance-based** methods. Features with larger numeric ranges will dominate distance calculations unless we standardize them first.

`StandardScaler` transforms each feature to have mean ≈ 0 and standard deviation = 1.

In [ ]:
# Step 1: Assemble raw features into a vector
assembler = VectorAssembler(inputCols=audio_features, outputCol="raw_features")

# Step 2: Scale the features (mean=0, std=1)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

# Fit and transform
assembled = assembler.transform(df)
scaler_model = scaler.fit(assembled)
scaled_df = scaler_model.transform(assembled)

print("✓ Features assembled and scaled")
scaled_df.select("track_name", "track_genre", "scaled_features").show(5, truncate=40)

## Part 4: PCA — Dimensionality Reduction

We have 10 features. PCA finds new axes (principal components) ordered by how much variance they explain. The first few components often capture most of the structure.

### Step 1: Fit PCA with all 10 components

We fit all 10 first to see the variance explained by each — then decide how many to keep.

In [ ]:
# YOUR CODE HERE:
# 1. Create a PCA object with k=10, inputCol="scaled_features", outputCol="pca_features"
# 2. Fit it on scaled_df
# 3. Extract the explained variance from the fitted model
#



In [ ]:
# Print variance explained by each component
cumulative = 0
print(f"{'PC':>4} {'Variance':>10} {'Cumulative':>12}")
print("-" * 30)
for i, v in enumerate(explained_variance):
    cumulative += v
    print(f"PC{i+1:>2} {v:>10.4f} {cumulative:>12.4f}")

### Step 2: Scree Plot

A scree plot shows the variance explained by each principal component. Look for the "elbow" — the point where adding more components gives diminishing returns.

In [ ]:
# YOUR CODE HERE:
# Create a scree plot with two lines:
# 1. Individual variance explained per component (bar chart or line)
# 2. Cumulative variance explained (line)
#




**Question:** How many components do you need to capture ~80% of the variance? Is there a clear elbow?

### Step 3: Transform with chosen number of components

Based on the scree plot, we'll reduce to a smaller number of components for clustering.

In [ ]:
# YOUR CODE HERE:
# 1. Pick k (number of components) based on your scree plot
# 2. Create a new PCA with that k
# 3. Fit and transform scaled_df
#
# Hint:
#   n_components = ???   # your choice


### PCA 2D Scatter (for comparison with t-SNE later)

Before clustering, let's see the PCA projection in 2D. This gives us a baseline to compare against t-SNE later.

In [ ]:
# YOUR CODE HERE:
# Extract PC1 and PC2 for a quick scatter plot
# NOTE: PySpark ML vectors don't support bracket indexing [0], [1].
# You must convert to array first using vector_to_array().
#
# Hint:
#   from pyspark.ml.functions import vector_to_array
#   pca_2d = pca_df.withColumn("pca_array", vector_to_array("pca_features")).select(
#       F.col("pca_array")[0].alias("PC1"),
#       F.col("pca_array")[1].alias("PC2"),
#       "track_genre"
#   ).sample(fraction=5000/pca_df.count(), seed=42).toPandas()
#
# Then plot: plt.scatter(pca_2d["PC1"], pca_2d["PC2"], s=3, alpha=0.3)

## Part 5: K-Means Clustering

K-Means assigns each point to the nearest centroid, then moves centroids to the center of their assigned points. Repeat until convergence.

**Problem:** You must choose *K* upfront. Two tools help:
1. **Elbow method** — plot WSSSE (within-cluster sum of squared errors) vs. K
2. **Silhouette score** — measures how similar points are to their own cluster vs. neighboring clusters (higher = better)

### Step 1: Elbow Method

In [ ]:
# YOUR CODE HERE:
# Try K = 2, 3, 4, 5, 6, 8, 10, 12, 15
# For each K:
#   1. Create KMeans(k=K, featuresCol="pca_features", seed=42)
#   2. Fit on pca_df
#   3. Record the WSSSE: model.summary.trainingCost
#
# Hint:
#   k_values = [2, 3, 4, 5, 6, 8, 10, 12, 15]
#   costs = []
#   for k in k_values:
#       km = KMeans(k=k, featuresCol="pca_features", seed=42)
#       km_model = km.fit(pca_df)
#       costs.append(km_model.summary.trainingCost)



In [ ]:
# Plot the elbow curve
plt.figure(figsize=(8, 5))
plt.plot(k_values, costs, 'bo-')
plt.xlabel("K (number of clusters)")
plt.ylabel("WSSSE (within-cluster sum of squared errors)")
plt.title("Elbow Method for Optimal K")
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

### Step 2: Silhouette Score

The silhouette score ranges from -1 to 1:
- **Near 1:** Points are well-matched to their own cluster
- **Near 0:** Points are on the boundary between clusters
- **Negative:** Points may be assigned to the wrong cluster

In [ ]:
# YOUR CODE HERE:
# For each K, compute the silhouette score using ClusteringEvaluator
#
# Hint:
#   evaluator = ClusteringEvaluator(
#       featuresCol="pca_features",
#       metricName="silhouette",
#       distanceMeasure="squaredEuclidean"
#   )
#   silhouettes = []
#   for k in k_values:
#       km = KMeans(k=k, featuresCol="pca_features", seed=42)
#       km_model = km.fit(pca_df)
#       predictions = km_model.transform(pca_df)
#       score = evaluator.evaluate(predictions)
#       silhouettes.append(score)
#       print(f"K={k:2d} → Silhouette: {score:.4f}")



In [ ]:
# Plot silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouettes, 'rs-')
plt.xlabel("K (number of clusters)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs. K")
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

**Question:** Do the elbow method and silhouette score agree on the best K? If not, which one would you trust more and why?

### Step 3: Fit Final K-Means Model

Pick your best K and fit the final model.

In [ ]:
# YOUR CODE HERE:
# 1. Pick K based on your analysis above
# 2. Train final K-Means model
# 3. Add cluster labels to the dataframe
#
# Hint:
#   best_k = ???
#   km_final = KMeans(k=best_k, featuresCol="pca_features", seed=42)
#   km_model = km_final.fit(pca_df)
#   clustered_df = km_model.transform(pca_df)



In [ ]:
# How many tracks in each cluster?
clustered_df.groupBy("prediction").count().orderBy("prediction").show()

## Part 6: Visualization with t-SNE

PCA is a linear method — it preserves global structure but can miss non-linear patterns. t-SNE is a non-linear technique designed specifically for **2D visualization**.

**Important:** t-SNE is for visualization only — never use it as input to another model. It is also O(n²), so we sample first.

We use scikit-learn for t-SNE since PySpark does not have a built-in implementation.

In [ ]:
from sklearn.manifold import TSNE

# Sample 5000 rows and convert to pandas
sample_df = clustered_df.select(
    *audio_features, "track_genre", "prediction", "scaled_features"
).sample(fraction=5000/clustered_df.count(), seed=42)

# Extract the scaled feature vectors as a numpy array
pdf = sample_df.select(*audio_features, "track_genre", "prediction").toPandas()
X_sample = pdf[audio_features].values

print(f"Sample size: {len(pdf)}")

In [ ]:
# YOUR CODE HERE:
# 1. Standardize X_sample (sklearn StandardScaler or use the values directly since they're already scaled)
# 2. Run t-SNE to reduce to 2 dimensions
# 3. Store the result in pdf["tsne_x"] and pdf["tsne_y"]
#
# Hint:
#   tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
#   tsne_result = tsne.fit_transform(X_sample)
#   pdf["tsne_x"] = tsne_result[:, 0]
#   pdf["tsne_y"] = tsne_result[:, 1]



In [ ]:
# Plot 1: t-SNE colored by K-Means cluster
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

scatter1 = axes[0].scatter(pdf["tsne_x"], pdf["tsne_y"],
                           c=pdf["prediction"], cmap="tab10", s=5, alpha=0.6)
axes[0].set_title("t-SNE colored by K-Means Cluster")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
plt.colorbar(scatter1, ax=axes[0], label="Cluster")

# Plot 2: t-SNE colored by genre (pick top 8 genres for readability)
top_genres = pdf["track_genre"].value_counts().head(8).index.tolist()
genre_subset = pdf[pdf["track_genre"].isin(top_genres)].copy()
genre_codes = {g: i for i, g in enumerate(top_genres)}
genre_subset["genre_code"] = genre_subset["track_genre"].map(genre_codes)

scatter2 = axes[1].scatter(genre_subset["tsne_x"], genre_subset["tsne_y"],
                           c=genre_subset["genre_code"], cmap="tab10", s=5, alpha=0.6)
axes[1].set_title("t-SNE colored by Genre (top 8)")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

# Legend for genres
handles = [plt.Line2D([0], [0], marker='o', color='w',
           markerfacecolor=plt.cm.tab10(i/10), markersize=8, label=g)
           for i, g in enumerate(top_genres)]
axes[1].legend(handles=handles, loc="best", fontsize=8)

plt.tight_layout()
plt.show()

**Interpret:** Compare the two plots side by side.
- Do the K-Means clusters map cleanly onto genres?
- Which genres overlap? Which are clearly separated?
- What does this tell you about whether "genre" is a good label for audio similarity?

## Part 7: Cluster Analysis

Let's look at what each cluster actually represents in terms of audio features.

In [ ]:
# YOUR CODE HERE:
# Compute the mean of each audio feature per cluster
#
# Hint:
#   cluster_profiles = clustered_df.groupBy("prediction").agg(
#       *[F.round(F.avg(f), 3).alias(f) for f in audio_features]
#   ).orderBy("prediction")
#   cluster_profiles.show(truncate=False)



**Interpret:** Can you name each cluster based on its audio profile?
- High energy + high loudness + low acousticness = ?
- High acousticness + high instrumentalness + low energy = ?
- High speechiness + high danceability = ?

Give each cluster a descriptive label.

In [ ]:
# What genres are most common in each cluster?
genre_by_cluster = clustered_df.groupBy("prediction", "track_genre").count()

# For each cluster, show the top 5 genres
from pyspark.sql.window import Window

w = Window.partitionBy("prediction").orderBy(F.desc("count"))
top_genres_per_cluster = genre_by_cluster.withColumn(
    "rank", F.row_number().over(w)
).filter(F.col("rank") <= 5)

top_genres_per_cluster.orderBy("prediction", "rank").show(50, truncate=False)

### Genre Purity

What percentage of each cluster belongs to its most common genre? High purity means K-Means discovered genre boundaries without seeing labels.

In [ ]:
# Genre purity: what percentage of each cluster's tracks belong to its top genre?
cluster_sizes = clustered_df.groupBy("prediction").count().withColumnRenamed("count", "total")

top_genre = genre_by_cluster.withColumn(
    "rank", F.row_number().over(Window.partitionBy("prediction").orderBy(F.desc("count")))
).filter(F.col("rank") == 1).select("prediction", "track_genre", "count")

purity = top_genre.join(cluster_sizes, "prediction").withColumn(
    "purity", F.round(F.col("count") / F.col("total") * 100, 1)
)

purity.select("prediction", "track_genre", "purity").orderBy("prediction").show(truncate=False)

### Radar Chart — Cluster Audio Profiles

A radar chart shows the mean audio features per cluster, normalized to 0–1. This makes it easy to visually identify what kind of music each cluster represents.

In [ ]:
# YOUR CODE HERE:
# 1. Get cluster_profiles as a pandas DataFrame
# 2. Normalize each feature column to 0-1 range (min-max)
# 3. Create a radar chart with one line per cluster
#
# Hint:
#   cp = cluster_profiles.toPandas()
#   features = audio_features
#   for f in features:
#       cp[f] = (cp[f] - cp[f].min()) / (cp[f].max() - cp[f].min())
#
#   angles = np.linspace(0, 2*np.pi, len(features), endpoint=False).tolist()
#   angles += angles[:1]  # close the polygon
#
#   fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
#   for _, row in cp.iterrows():
#       values = [row[f] for f in features] + [row[features[0]]]
#       ax.plot(angles, values, "o-", label=f"Cluster {int(row.prediction)}")
#   ax.set_xticks(angles[:-1])
#   ax.set_xticklabels(features, fontsize=8)
#   ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
#   plt.tight_layout()
#   plt.show()

### Genre Map — All 114 Genres

Now we zoom out. Using the **full dataset** (`df_full`, all 114 genres), compute the mean audio profile per genre and run t-SNE on just 114 points. Each dot is a genre. Genres that sound similar land next to each other.

We exclude `duration_ms` — it describes track length, not what the music *sounds like*.

In [ ]:
# Genre-level t-SNE map — using df_full (all 114 genres)
from sklearn.preprocessing import StandardScaler as SkScaler

sound_features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

# Compute mean audio profile per genre from FULL dataset
genre_profiles = df_full.groupBy("track_genre").agg(
    *[F.avg(f).alias(f) for f in sound_features]
).toPandas()

# Standardize
X_genres = SkScaler().fit_transform(genre_profiles[sound_features].values)

# t-SNE on 114 points — instant
tsne_genres = TSNE(n_components=2, random_state=42, perplexity=10, max_iter=1000)
genre_2d = tsne_genres.fit_transform(X_genres)

# Plot with labels
fig, ax = plt.subplots(figsize=(16, 12), dpi=150)
ax.scatter(genre_2d[:, 0], genre_2d[:, 1], s=10, alpha=0.7, c="steelblue", zorder=5)

for i, genre in enumerate(genre_profiles["track_genre"]):
    ax.annotate(genre, (genre_2d[i, 0], genre_2d[i, 1]),
                fontsize=7, alpha=0.85, ha="center", va="bottom",
                textcoords="offset points", xytext=(0, 3))

ax.set_title("Genre Map — t-SNE on Mean Audio Profiles (no duration, no genre labels)", fontsize=13)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

## Part 8: Discussion Questions

Answer these in a markdown cell below:

1. **Scaling:** What would happen if you ran PCA and K-Means on the raw (unscaled) features? Which features would dominate and why?

2. **PCA before K-Means:** Why did we apply PCA before K-Means instead of clustering on all 10 features? Connect this to the curse of dimensionality from the reading.

3. **Clusters vs. genres:** Do your clusters correspond to genres? If not, what does this suggest — are the clusters wrong, or is the genre labeling noisy?

4. **t-SNE vs. PCA:** The t-SNE plot likely shows more separation between groups than a 2D PCA scatter would. Why can t-SNE reveal structure that PCA misses? Why can't you use t-SNE output as input to K-Means?

In [ ]:
# YOUR ANSWERS HERE:


In [ ]:
spark.stop()